<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/multi_company_financial_dashboard_bokeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

from math import pi

from bokeh.io import curdoc
from bokeh.io import output_notebook, show
from bokeh.layouts import row, column
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select
)

In [ ]:
output_notebook()

In [ ]:
BACKGROUND = "#0B1120"
CARD = "#1E293B"
TEXT = "#F8FAFC"
GRID = "#334155"

WIDTH = 650
HEIGHT = 400

In [ ]:
companies = {
    "NVIDIA":"NVDA",
    "Tesla":"TSLA",
    "Amazon":"AMZN",
    "Meta":"META",
    "Netflix":"NFLX",
    "AMD":"AMD"
}

In [ ]:
ticker = companies['NVIDIA']

df = yf.download(
    ticker,
    period = "2y"
)

/tmp/ipykernel_10379/860481884.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


In [ ]:
df.reset_index(inplace=True)

In [ ]:
df.columns = df.columns.get_level_values(0)

In [ ]:
df.head()

Price,Date,Close,High,Low,Open,Volume
0,2024-05-08,90.361015,91.142572,89.369570,89.432538,325721000
1,2024-05-09,88.696945,91.020631,88.181237,90.477938,378013000
2,2024-05-10,89.827309,91.349452,89.176675,90.254070,335325000
3,2024-05-13,90.348022,90.946684,88.479074,90.426973,289680000
4,2024-05-14,91.304482,91.599313,88.883843,89.548469,296507000


In [ ]:
df['Returns'] = df['Close'].pct_change()

In [ ]:
df['MA20'] = df['Close'].rolling(20).mean()
df['MA50'] = df['Close'].rolling(50).mean()

In [ ]:
df['Volatility'] = df['Returns'].rolling(20).std()

In [ ]:
df['Upper'] = df['MA20'] + 2 * df['Volatility']
df['Lower'] = df['MA20'] - 2 * df['Volatility']

In [ ]:
investment = 10000

shares = investment / df['Close'].iloc[0]

In [ ]:
df['Portfolio'] = shares * df['Close']

In [ ]:
df.dropna(inplace=True)

In [ ]:
source = ColumnDataSource()

latest_price = round(
    df['Close'].iloc[-1],
    2
)

In [ ]:
daily_return = round(
    df['Returns'].iloc[-1] * 100,
    2
)

In [ ]:
volatility = round(
    df['Volatility'].mean() * 100,
    2
)

In [ ]:
portfolio_value = round(
    df['Portfolio'].iloc[-1],
    2
)

In [ ]:
profit = round(
    portfolio_value - investment,
    2
)

In [ ]:
risk_free_rate = 0.02

sharpe_ratio = round(
    (
        df['Returns'].mean()*252
        - risk_free_rate
    )
    /
    (
        df['Returns'].std()
        * np.sqrt(252)
    ),
    2
)

In [ ]:
def create_kpi(title, value, color):

    return Div(text=f"""
    <div style="
    background:{color};
    padding:20px;
    border-radius:18px;
    text-align:center;
    width:220px;
    height:120px;
    box-shadow:0px 4px 15px rgba(0,0,0,0.5);
    ">

    <h2 style="
    color:white;
    font-family:Arial;
    ">
    {title}
    </h2>

    <h1 style="
    color:white;
    ">
    {value}
    </h1>

    </div>
    """)

In [ ]:
kpi1 = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi(
    "Profit/Loss",
    f"${profit}",
    '#EA580C'
)

kpi6 = create_kpi(
    "Sharpe Ratio",
    sharpe_ratio,
    "#0891b2"
)

In [ ]:
company_selector = Select(
    title="Select Company",
    value = "NVIDIA",
    options = list(companies.keys())
)

In [ ]:
inc = df['Close'] > df['Open']
dec = df['Close'] < df['Open']

In [ ]:
w = 12 * 60 * 60 * 1000

TOOLS = "pan,wheel_zoom,box_zoom,reset,save"

p1 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Candlestick Analysis',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

In [ ]:
p1.segment(
    df['Date'],
    df['High'],
    df['Date'],
    df['Low'],
    color = 'white'
)

GlyphRenderer(id='p1079', ...)

In [ ]:
p1.vbar(
    df.loc[inc,'Date'],
    w,
    df.loc[inc,'Open'],
    df.loc[inc,'Close'],
    fill_color = "#10B981",
    line_color = "#10B981"
)

p1.vbar(
    df.loc[dec,'Date'],
    w,
    df.loc[dec,'Open'],
    df.loc[dec,'Close'],
    fill_color = '#EF4444',
    line_color = '#EF4444'
)

GlyphRenderer(id='p1099', ...)

In [ ]:
p2 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Moving Average Analytics',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

In [ ]:
p2.line(
    df['Date'],
    df['Close'],
    line_width = 2,
    color = 'white',
    legend_label = 'Close Price'
)

p2.line(
    df['Date'],
    df['MA20'],
    line_width=3,
    color = '#3882F6',
    legend_label = 'MA20'
)

p2.line(
    df['Date'],
    df['MA50'],
    line_width=3,
    color = '#F59E0B',
    legend_label = 'MA50'
)

GlyphRenderer(id='p1180', ...)

In [ ]:
p3 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Volume Analysis',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

p3.vbar(
    x = df['Date'],
    top = df['Volume'],
    width = w,
    color = '#06B6D4'
)

GlyphRenderer(id='p1241', ...)

In [ ]:
p4 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Risk & Volatility',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND

)

p4.line(
    df['Date'],
    df['Volatility'],
    line_width = 3,
    color = '#EF4444'
)

GlyphRenderer(id='p1301', ...)

In [ ]:
hist, edges = np.histogram(
    df['Returns'].dropna(),
    bins=40
)

p5 = figure(
    width = WIDTH,
    height = HEIGHT,
    title = ' Return Distribution',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

p5.quad(
    top=hist,
    bottom=0,
    left = edges[:-1],
    right = edges[1:],
    fill_color = '#8B5CF6',
    line_color = 'white'
)

GlyphRenderer(id='p1347', ...)

In [ ]:
p6 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Portfolio Growth',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

p6.line(
    df['Date'],
    df['Portfolio'],
    line_width = 4,
    color = "#22C55E"
)

GlyphRenderer(id='p1407', ...)

In [ ]:
plots = [p1,p2,p3,p4,p5,p6]

for p in plots:
  p.title.text_color = TEXT

  p.xaxis.major_label_text_color = TEXT
  p.yaxis.major_label_text_color = TEXT

  p.xaxis.axis_label_text_color = TEXT
  p.yaxis.axis_label_text_color = TEXT

  p.grid.grid_line_color = GRID

In [ ]:
title = Div(text="""
<div style="
background:linear-gradient(
90deg,
#111827,
#1D4ED8
);

padding:25px;
border-radius:18px;
box-shadow:0px 4px 15px rgba(0,0,0,0.5);
">

<h1 style="
color:white;
text-align:center;
font-size:40px;
font-family:Arial;
">
MULTI-COMPANY FINANCIAL ANALYTICS DASHBOARD
</h1>

</div>
""")

In [ ]:
def update(attr, old, new):
  company = company_selector.value
  ticker = companies[company]

  new_data = yf.download(
      ticker,
      period = '2y'
  )

  new_data.reset_index(inplace=True)

  new_data.columns = (
      new_data.columns.get_level_values(0)
  )

  new_data['Returns'] = (
      new_data['Close'].pct_change()
  )

  new_data['MA20'] = (
      new_data['Close'].rolling(20).mean()
  )

  new_data['MA50'] = (
      new_data['Close'].rolling(50).mean()
  )

  new_data['Volatility'] = (
      new_data['Returns'].rolling(20).std()
  )

  new_data.dropna(inplace=True)

  source.data = ColumnDataSource(
      new_data
  ).data

  company_selector.on_change(
      'value',
      update
  )

In [ ]:
dashboard = column(

    title,

    row(
        kpi1,
        kpi2,
        kpi3
    ),

    row(
        kpi4,
        kpi5,
        kpi6
    ),

    row(
        company_selector
    ),

    row(
        p1,
        p2
    ),

    row(
        p3,
        p4
    ),

    row(
        p5,
        p6
    )
)


In [ ]:
show(dashboard)